## Modern LLM RL Paradigm (PPO, KL Penalties, GRPO Mechanics, RLVR, and Reward Hacking)

---

Now we connect classical policy gradients to how state-of-the-art LLM reasoning models are trained.

---

### Reward Hacking

---

## 1. The Catastrophic Drift Problem and the KL Penalty

If we naively optimize an LLM using only policy gradients, the model quickly discovers that it can exploit edge cases:

- generating repetitive garbage tokens that confuse reward models
- forgetting basic language syntax and conversational ability (catastrophic forgetting)
- collapsing into a narrow, repetitive mode

To prevent the active policy $\pi_\theta$ from drifting too far from the original base model $\pi_{\text{ref}}$ (the starting SFT checkpoint), we enforce a Kullback–Leibler (KL) divergence penalty:

$$
D_{\text{KL}}(\pi_\theta \parallel \pi_{\text{ref}}) = \sum_{a \in \mathcal{V}} \pi_\theta(a \mid s) \log \left( \frac{\pi_\theta(a \mid s)}{\pi_{\text{ref}}(a \mid s)} \right)
$$

### Key terms

- $\pi_\theta$: the active actor policy being trained
- $\pi_{\text{ref}}$: a frozen copy of the original model

### Effect

If $\pi_\theta$ assigns high probability to a token that $\pi_{\text{ref}}$ thought was virtually impossible, the ratio shoots up, and a penalty is subtracted from the reward.


## 2. PPO vs. GRPO (The Memory Bottleneck Shift)

In classical Proximal Policy Optimization (PPO) for LLMs, you need 4 separate models in GPU VRAM:

- Actor ($\pi_\theta$): the policy being trained (trainable)
- Critic / value network ($V_\phi$): estimates expected future return from state $s$ to compute baselines (trainable)
- Reference model ($\pi_{\text{ref}}$): frozen base model for KL computation (frozen)
- Reward model ($R_\psi$): scores the generation (frozen)

Having both an actor and a critic of large parameter sizes (for example, 7B–70B) creates an enormous memory footprint.


## 3. Enter GRPO (Group Relative Policy Optimization)

Introduced in the DeepSeekMath / DeepSeek-R1 line of research, GRPO eliminates the critic/value model entirely.

Instead of training a separate value network to predict a baseline:

1. For a single prompt $q$, sample a group of $G$ independent outputs: $\{o_1, o_2, \dots, o_G\}$.
2. Evaluate the scalar reward for each output: $\{r_1, r_2, \dots, r_G\}$.
3. Normalize rewards across the group to compute the advantage:

$$
A_i = \frac{r_i - \text{mean}(\{r_1, \dots, r_G\})}{\text{std}(\{r_1, \dots, r_G\}) + \epsilon}
$$

### Flow

```text
Prompt q ───┬───> Sample o_1 ───> Reward r_1 ───┐
            ├───> Sample o_2 ───> Reward r_2 ───┼───> Group Normalization ───> Advantages A_i
            ├───> Sample o_3 ───> Reward r_3 ───┤
            └───> Sample o_4 ───> Reward r_4 ───┘
```

By computing the baseline directly from group statistics, you save roughly 50% of the training memory, allowing larger batch sizes or training larger models on fewer GPUs.

## 4. RLVR (Reinforcement Learning with Verifiable Rewards)

Rather than using an imperfect, learned neural reward model that can be gamed, RLVR uses deterministic, programmatic verifiers:

- Math: extract the final boxed answer and match it against the ground truth using a symbolic math parser such as SymPy
- Coding: run generated code in a sandbox and execute deterministic unit tests (pass = $1$, fail = $0$)
- Format: check whether the output strictly contains <think>...</think> and <answer>...</answer> tags

Because the reward is mathematically or programmatically verifiable, the model cannot easily game the score without actually solving the underlying problem.
